In [9]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access the API key from environment variables
groq_api_key = os.getenv("GROQ_API_KEY")

In [12]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import trim_messages


# Initialize Groq LLM
llm = ChatGroq(api_key=groq_api_key, model="llama-3.3-70b-versatile", temperature=0)

# Store for session histories
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Create runnable with message history
with_message_history = RunnableWithMessageHistory(llm, get_session_history)
config = {"configurable": {"session_id": "chat1"}}

# Define prompt template
prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a customer support agent at Tailorsin.com, a fast e-tailoring company "
                              "that collects cloth material, takes measurements, stitches, and delivers garments in 24 hours. "
                              "Answer the customer questions to the best of your ability."),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

# Create the chain
chain = prompt_template | llm | StrOutputParser()

# Function to chat with history
def chat_with_bot(user_input, session_id="chat1"):
    config = {"configurable": {"session_id": session_id}}
    
    # Invoke the chain with message history
    response = with_message_history.invoke(
        [HumanMessage(content=user_input)],
        config=config
    )
    
    return response.content

# Test the chatbot
if __name__ == "__main__":
    # Example conversation
    response1 = chat_with_bot("Hello, I have a question about my order.")
    print("Bot:", response1)
    
    response2 = chat_with_bot("When will my order be delivered?")
    print("Bot:", response2)

# Optional: Message trimming for managing conversation history
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",  # Fixed: removed space, added quotes
    token_counter=llm,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

# Example of using trimmer (optional)
def chat_with_trimming(user_input, session_id="chat1"):
    config = {"configurable": {"session_id": session_id}}
    
    # Get history
    history = get_session_history(session_id)
    
    # Trim messages if needed
    trimmed_messages = trimmer.invoke(history.messages)
    
    # Use trimmed messages in your chain
    response = chain.invoke({"messages": trimmed_messages + [HumanMessage(content=user_input)]})
    
    return response

Bot: I'd be happy to help with your question. Can you please provide more details about your order, such as the order number, what you ordered, and what's concerning you? This will help me better understand the issue and provide a more accurate response.
Bot: I'm a large language model, I don't have direct access to your order information or the ability to track your order. I'm here to provide general assistance and answer questions to the best of my knowledge.

To get an update on your order delivery, I recommend checking the following options:

1. **Check your email**: Look for an email from the seller or shipping carrier with tracking information and estimated delivery dates.
2. **Visit the seller's website**: Log in to your account on the seller's website and check the order status.
3. **Contact the seller's customer support**: Reach out to the seller's customer support team directly via phone, email, or live chat to ask about the status of your order.
4. **Track your order**: If y

In [1]:
import requests

url = "https://crm.tailorsin.com/tailorsin-api/api/customers"

params = {
    "customer_id": 38443
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json())

200
{'status': 'success', 'data': {'id': '38443', 'uid': None, 'is_client': None, 'toggle': '0', 'salute': None, 'b2b': None, 'cname': 'shweta yallatti', 'dob': '0000-00-00', 'clgstin': '', 'tel': None, 'email': None, 'whatsapp': '917259213560', 'address': None, 'address2': None, 'locality': None, 'city': None, 'pincode': None, 'lat': None, 'lng': None, 'add_date': '2026-03-21 18:21:03', 'chat': '0', 'new_whatsapp': '0', 'last_spoken': '1774165426', 'deleted': None, 'otp_auth': None, 'telcal': None, 'fasdis': None, 'on_quote': None, 'welcome_sent': None, 'brand': None, 'website': None, 'lang1': '', 'lang2': '', 'chatby': 'Meghana', 'client_store_id': '0', 'clientsource': None, 'instaid': None, 'contact_person': '', 'bank_ac_name': '', 'bank_ac_no': '', 'bank_ifsc': '', 'bank_address': '', 'isvendor': '0'}}


In [9]:
import requests

url = "https://crm.tailorsin.com/tailorsin-api/api/customers"

params = {
    "mobile": "7678250097"
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.text)





200
{"status":"success","data":{"id":"38715","uid":null,"is_client":null,"toggle":"0","salute":null,"b2b":null,"cname":"Sana Islam","dob":"0000-00-00","clgstin":"","tel":null,"email":null,"whatsapp":"917678250097","address":null,"address2":null,"locality":null,"city":null,"pincode":null,"lat":null,"lng":null,"add_date":"2026-03-30 12:26:30","chat":"0","new_whatsapp":"0","last_spoken":"1774858845","deleted":null,"otp_auth":null,"telcal":null,"fasdis":null,"on_quote":null,"welcome_sent":null,"brand":null,"website":null,"lang1":"","lang2":"","chatby":"Meghana","client_store_id":"0","clientsource":null,"instaid":null,"contact_person":"","bank_ac_name":"","bank_ac_no":"","bank_ifsc":"","bank_address":"","isvendor":"0"}}
